In [ ]:
!nvidia-smi

Sun Aug  2 11:13:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install -U ultralytics==8.4.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.8 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# paths
ZIP_PATH = "/content/drive/MyDrive/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph.zip"
WORK_DIR = "/content"

# copy & unzip
!cp "$ZIP_PATH" $WORK_DIR/data.zip
!unzip -o $WORK_DIR/data.zip -d $WORK_DIR

Archive:  /content/data.zip
   creating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/
   creating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/classes.txt  
   creating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/00001.jpg  
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/000010.jpg  
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/0000100.jpg  
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/00001000.jpg  
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/00001001.jpg  
  inflating: /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/images/00001002.jpg  
  inflating: /content/MOPG-7 A

In [ ]:
import glob, os, random, shutil

random.seed(42)

SRC_ROOT = "/content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset"          # root containing your extracted images (+ labels, if detection)
SPLIT_ROOT = "/content/Data_split"  # new root with clean train/val/test folders

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# Recursively find every image under SRC_ROOT, wherever it lives in the extracted zip
all_images = [p for p in glob.glob(os.path.join(SRC_ROOT, "**", "*"), recursive=True)
              if p.lower().endswith(IMG_EXTS)]
assert len(all_images) > 0, f"No images found under {SRC_ROOT} — check ZIP_PATH extraction / folder names."
print(f"Found {len(all_images)} images under {SRC_ROOT}")

# Try to find a matching YOLO-format label (.txt) for each image, if this is a detection task.
# Label is assumed to sit in a parallel 'labels' folder with the same filename stem.
def find_label(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    candidates = glob.glob(os.path.join(SRC_ROOT, "**", stem + ".txt"), recursive=True)
    # Prefer a candidate that lives under a "labels" folder if there are multiple hits
    labels_only = [c for c in candidates if os.sep + "labels" in c]
    return (labels_only or candidates or [None])[0]

pairs = [(img, find_label(img)) for img in all_images]
n_with_labels = sum(1 for _, l in pairs if l)
print(f"{n_with_labels}/{len(pairs)} images have a matching label file "
      f"({'detection task' if n_with_labels else 'no labels found — treating as classification/no-label set'})")

random.shuffle(pairs)
n = len(pairs)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
splits = {
    "train": pairs[:n_train],
    "val": pairs[n_train:n_train + n_val],
    "test": pairs[n_train + n_val:],
}
for name, s in splits.items():
    print(f"{name}: {len(s)} images")

for split_name, split_pairs in splits.items():
    img_dir = os.path.join(SPLIT_ROOT, "images", split_name)
    lbl_dir = os.path.join(SPLIT_ROOT, "labels", split_name)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    for img_path, lbl_path in split_pairs:
        shutil.copy2(img_path, os.path.join(img_dir, os.path.basename(img_path)))
        if lbl_path:
            shutil.copy2(lbl_path, os.path.join(lbl_dir, os.path.basename(lbl_path)))

print("Split complete. No image appears in more than one of train/val/test.")


Found 2095 images under /content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset
2095/2095 images have a matching label file (detection task)
train: 1676 images
val: 209 images
test: 210 images
Split complete. No image appears in more than one of train/val/test.


In [ ]:
classes_path = "/content/MOPG-7 A Multi-Clinic Dental Panoramic Radiograph/Dataset/classes.txt"
yaml_path = "/content/data.yaml"

# Read class names
with open(classes_path, "r") as f:
    classes = [line.strip() for line in f if line.strip()]

# Create data.yaml
with open(yaml_path, "w") as f:
    f.write(f"path: {SPLIT_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n\n")
    f.write(f"nc: {len(classes)}\n")
    f.write("names:\n")
    for i, c in enumerate(classes):
        f.write(f"  {i}: {c}\n")

print("Classes:", classes)

print("\nGenerated data.yaml:\n")
with open(yaml_path, "r") as f:
    print(f.read())

Classes: ['Missing', 'Dental Crown', 'Root Canal', 'Caries', 'Broken Down', 'Wisdom Teeth', 'Healthy']

Generated data.yaml:

path: /content/Data_split
train: images/train
val: images/val
test: images/test

nc: 7
names:
  0: Missing
  1: Dental Crown
  2: Root Canal
  3: Caries
  4: Broken Down
  5: Wisdom Teeth
  6: Healthy



In [ ]:
!wget -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11m.pt
assert os.path.exists("yolo11m.pt"), "YOLOv11 weights missing!"


In [ ]:
model = YOLO("yolo11m.pt")
model.model.info()


YOLO11m summary: 231 layers, 20,114,688 parameters, 0 gradients, 68.5 GFLOPs


(231, 20114688, 0, 68.520192)

In [ ]:
results = model.train(
    model="yolo11m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov11m"
)


New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov11m, nbs=64, nms=Fal

In [ ]:
metrics_val_v11 = model.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv11m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v11.box.map50)
print("mAP@0.5:0.95:", metrics_val_v11.box.map)
print("Precision   :", metrics_val_v11.box.mp)
print("Recall      :", metrics_val_v11.box.mr)

Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3548.4±268.8 MB/s, size: 857.9 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 73.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.1it/s 2.7s
                   all        209        923      0.709      0.716      0.717      0.335
               Missing        118        258      0.465      0.446      0.434      0.128
          Dental Crown         55        169       0.82      0.941       0.89      0.457
            Root Canal         58        176      0.668      0.676      0.635      0.212
                Caries         89        151      0.692      0.642      0.681      0.237
           Broken Down        

In [ ]:
metrics_test_v11 = model.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv11m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v11.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v11.box.map:.4f}")
print(f"Precision    : {metrics_test_v11.box.mp:.4f}")
print(f"Recall       : {metrics_test_v11.box.mr:.4f}")

Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,035,429 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3411.2±1315.6 MB/s, size: 909.6 KB)
val: Scanning /content/Data_split/labels/test... 210 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 210/210 1.5Kit/s 0.1s
val: New cache created: /content/Data_split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 4.6it/s 3.0s
                   all        210       1088      0.729      0.714      0.704      0.334
               Missing        111        268      0.531      0.452      0.458      0.144
          Dental Crown         55        217      0.875      0.971      0.969      0.491
            Root Canal         67        244      0.706      0.746      0.661      0.227
                Caries         94        147      0.631      0

In [ ]:
model_v10 = YOLO("yolov10m.pt")  # auto-downloads official YOLOv10n weights
model_v10.model.info()


YOLOv10m summary: 288 layers, 16,576,768 parameters, 0 gradients, 64.5 GFLOPs


(288, 16576768, 0, 64.4704512)

In [ ]:
results = model_v10.train(
    model="yolov10m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov10m"
)

New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov10m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov10m, nbs=64, nms=Fa

In [ ]:
metrics_val_v10 = model_v10.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv10m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v10.box.map50)
print("mAP@0.5:0.95:", metrics_val_v10.box.map)
print("Precision   :", metrics_val_v10.box.mp)
print("Recall      :", metrics_val_v10.box.mr)


Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10m summary (fused): 136 layers, 15,317,221 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3379.2±1589.1 MB/s, size: 696.1 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 73.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.7it/s 2.5s
                   all        209        923       0.66      0.742      0.724      0.344
               Missing        118        258      0.441       0.44      0.401      0.117
          Dental Crown         55        169      0.767      0.976      0.933      0.501
            Root Canal         58        176      0.682      0.722      0.705       0.23
                Caries         89        151      0.638      0.603      0.603      0.215
           Broken Down      

In [ ]:
metrics_test_v10 = model_v10.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv10m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v10.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v10.box.map:.4f}")
print(f"Precision    : {metrics_test_v10.box.mp:.4f}")
print(f"Recall       : {metrics_test_v10.box.mr:.4f}")

Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv10m summary (fused): 136 layers, 15,317,221 parameters, 0 gradients, 58.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2636.9±1807.0 MB/s, size: 411.8 KB)
val: Scanning /content/Data_split/labels/test.cache... 210 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 210/210 88.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.1it/s 2.8s
                   all        210       1088      0.633      0.715      0.678      0.323
               Missing        111        268      0.468      0.433      0.386       0.12
          Dental Crown         55        217      0.849      0.968      0.955      0.508
            Root Canal         67        244       0.65      0.717      0.676       0.23
                Caries         94        147      0.539      0.442      0.455      0.148
           Broken Down     

In [ ]:
model_v8 = YOLO("yolov8m.pt")  # auto-downloads official YOLOv8m weights
model_v8.model.info()

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


(169, 25902640, 0, 79.313664)

In [ ]:
results_v8 = model_v8.train(
    model="yolo8m.pt",
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    box=9.0,
    dfl=2.0,
    cls=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=5.0,
    scale=0.5,
    project="runs/scale_adaptive",
    name="sa_yolov8"
)


New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=9.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.7, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sa_yolov82, nbs=64, nms=False

In [ ]:
metrics_val_v8 = model_v8.val(
    data="/content/data.yaml",
    split="val",
    imgsz=640,
    device=0
)

print("=== YOLOv8m VALIDATION RESULTS ===")
print("mAP@0.5     :", metrics_val_v8.box.map50)
print("mAP@0.5:0.95:", metrics_val_v8.box.map)
print("Precision   :", metrics_val_v8.box.mp)
print("Recall      :", metrics_val_v8.box.mr)


Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3270.6±1511.9 MB/s, size: 696.1 KB)
val: Scanning /content/Data_split/labels/val.cache... 209 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 209/209 73.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 5.6it/s 2.5s
                   all        209        923      0.719      0.706      0.729      0.333
               Missing        118        258      0.455      0.539      0.458      0.131
          Dental Crown         55        169      0.839      0.958      0.914      0.482
            Root Canal         58        176      0.687      0.716      0.705      0.225
                Caries         89        151      0.694      0.675      0.687      0.246
           Broken Down         2

In [ ]:
metrics_test_v8 = model_v8.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    device=0
)

print("=== YOLOv8m TEST RESULTS ===")
print(f"mAP@0.5      : {metrics_test_v8.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics_test_v8.box.map:.4f}")
print(f"Precision    : {metrics_test_v8.box.mp:.4f}")
print(f"Recall       : {metrics_test_v8.box.mr:.4f}")

Ultralytics 8.4.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layers, 25,843,813 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2612.7±1246.0 MB/s, size: 869.3 KB)
val: Scanning /content/Data_split/labels/test.cache... 210 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 210/210 58.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 14/14 4.6it/s 3.1s
                   all        210       1088      0.739      0.712      0.717      0.337
               Missing        111        268      0.532      0.491       0.44      0.133
          Dental Crown         55        217      0.924      0.935      0.953      0.481
            Root Canal         67        244      0.726      0.716      0.645      0.218
                Caries         94        147      0.685      0.503      0.552      0.175
           Broken Down         

In [ ]:
import shutil

zip_path = "/content/runs.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content",
    base_dir="runs"
)

print("✅ runs folder zipped at:", zip_path)


✅ runs folder zipped at: /content/runs.zip


In [ ]:
# from google.colab import files

# files.download(yaml_path)

In [ ]:
import shutil
from google.colab import files
import os

# Define the path for the new zip file
full_content_zip_path = "/content/Split_Data.zip"

# Create a zip archive of the entire /content directory
shutil.make_archive(
    base_name=full_content_zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content/Data_split",
    base_dir="."
)

print(f"✅ All content zipped at: {full_content_zip_path}")

# Download the created zip file
files.download(full_content_zip_path)

✅ All content zipped at: /content/Split_Data.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files
import os

# Define the path for the new zip file
full_content_zip_path = "/content/Split_Data.zip"

# Create a zip archive of the entire /content directory
shutil.make_archive(
    base_name=full_content_zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/content/Data_split",
    base_dir="."
)

print(f"✅ All content zipped at: {full_content_zip_path}")

# Download the created zip file
files.download(full_content_zip_path)